# Step 5 — Convert boundary to H3 cells

**H3** is Uber's hierarchical hexagonal spatial indexing system.
This notebook converts your boundary polygon into a set of H3 cell IDs
that cover the area at a chosen resolution.

## Why H3 cells?

The duckOSM road network already has H3 indices on every edge (`from_cell`, `to_cell`).
Generating the boundary's H3 cells lets you:
- Partition routing queries by cell (faster lookups)
- Join the road network with other H3-indexed datasets (traffic counts, population, etc.)
- Visualise coverage in tools like **kepler.gl** (which renders H3 IDs natively)

## Resolution guide

| Resolution | Approx cell area | Cells for ~10 km² district |
|---|---|---|
| 7 | ~5 km² | ~2 |
| 8 | ~0.7 km² | ~15 |
| 9 | ~0.1 km² | ~100 |
| 10 | ~0.015 km² | ~650 |
| 15 | microscopic | millions |

Resolution **8** is a good default for district-level analysis.  
Match the resolution to the one used in duckOSM (`h3_resolution` in `config/*.yaml`).

In [ ]:
%pip install h3 folium --quiet

In [ ]:
%%time
import json
import yaml
import h3
import duckdb
import folium
import pandas as pd
import geopandas as gpd
from pathlib import Path
from shapely.geometry import shape, mapping

# ── Configuration ─────────────────────────────────────────────────────────
NAME        = 'sodermalm'
RESOLUTION  = 8                          # H3 resolution — match h3_resolution in config YAML
BUFFER_DEG  = 0.002                      # ~150-200 m buffer to cover boundary edges
BOUNDARY_FILE = Path(f'../boundaries/{NAME}.geojson')
OUTPUT_DIR  = Path('../output')
# ─────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(exist_ok=True)

# Read DB path from config YAML (generated by notebook 1)
CONFIG_PATH = Path(f'../config/{NAME}.yaml')
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        cfg = yaml.safe_load(f)
    DB_PATH = Path(cfg['output_path']) / f"{cfg['name']}.duckdb"
    # Use resolution from config if not overridden above
    config_res = cfg.get('options', {}).get('h3_resolution', RESOLUTION)
    if config_res != RESOLUTION:
        print(f'Note: config h3_resolution={config_res}, using RESOLUTION={RESOLUTION} from this cell')
else:
    DB_PATH = Path(f'../db/{NAME}.duckdb')
    print('WARNING: config not found — using default DB path')

print(f'Boundary  : {BOUNDARY_FILE}')
print(f'Resolution: {RESOLUTION}')
print(f'DuckDB    : {DB_PATH}')

---
## Convert polygon to H3 cells

We slightly **buffer** the boundary before converting.
Without a buffer, cells that touch the boundary edge are sometimes excluded,
leaving gaps around the perimeter.

`h3.polygon_to_cells()` requires coordinates in **(lat, lon)** order — the opposite
of GeoJSON which uses **(lon, lat)**. We swap them during conversion.

In [ ]:
%%time
# Load boundary
with open(BOUNDARY_FILE) as f:
    geojson_data = json.load(f)

feature  = geojson_data['features'][0]
geometry = feature['geometry']

# Buffer to ensure full edge coverage
shapely_poly   = shape(geometry)
buffered_poly  = shapely_poly.buffer(BUFFER_DEG)
buffered_geom  = mapping(buffered_poly)

# h3-py v4 expects (lat, lon) — swap from GeoJSON (lon, lat)
def swap_coords(coords):
    return [(lat, lon) for lon, lat in coords]

outer_ring = swap_coords(buffered_geom['coordinates'][0])
holes      = [swap_coords(ring) for ring in buffered_geom['coordinates'][1:]]

h3_polygon = h3.LatLngPoly(outer_ring, *holes)
cells      = list(h3.polygon_to_cells(h3_polygon, RESOLUTION))

print(f'Resolution {RESOLUTION}: {len(cells)} H3 cells cover {NAME}')

---
## Visualize

The boundary is shown in red, H3 cells in blue.  
Check that the cells fully cover the boundary without large gaps at the edges.

In [ ]:
%%time
# Compute map center from the original (unbuffered) boundary
coords = geometry['coordinates'][0]
center = [sum(c[1] for c in coords) / len(coords),
          sum(c[0] for c in coords) / len(coords)]

m = folium.Map(location=center, zoom_start=13, tiles='OpenStreetMap')

# Original boundary outline
folium.GeoJson(
    geojson_data,
    style_function=lambda _: {'color': 'red', 'weight': 2, 'fillOpacity': 0},
    tooltip='Boundary',
).add_to(m)

# H3 cells
for cell in cells:
    cell_boundary = h3.cell_to_boundary(cell)   # returns (lat, lon) pairs
    folium.Polygon(
        locations=cell_boundary,
        color='blue', weight=1,
        fill=True, fill_color='blue', fill_opacity=0.1,
        tooltip=cell,
    ).add_to(m)

m

---
## Save

Two output files:

| File | Use with |
|---|---|
| `{name}_{resolution}_h3_cells.csv` | **kepler.gl** — drag and drop, hexagons render automatically |
| `{name}_{resolution}_h3_cells.geojson` | QGIS, geopandas, any GIS tool |

The CSV uses column name `h3_id` which kepler.gl recognises automatically.

In [ ]:
%%time
import pandas as pd

# ── CSV (kepler.gl compatible) ────────────────────────────────────────────
csv_path = OUTPUT_DIR / f'{NAME}_{RESOLUTION}_h3_cells.csv'
pd.DataFrame({'h3_id': cells}).to_csv(csv_path, index=False)
print(f'Saved CSV     → {csv_path}  ({len(cells)} cells)')

# ── GeoJSON (hexagon polygons) ────────────────────────────────────────────
cell_polygons = []
for cell in cells:
    boundary_latlon = h3.cell_to_boundary(cell)     # (lat, lon) pairs
    coords = [(lon, lat) for lat, lon in boundary_latlon]
    coords.append(coords[0])                         # close ring
    cell_polygons.append({
        'type': 'Feature',
        'geometry': {'type': 'Polygon', 'coordinates': [coords]},
        'properties': {'h3_id': cell, 'resolution': RESOLUTION},
    })

geojson_path = OUTPUT_DIR / f'{NAME}_{RESOLUTION}_h3_cells.geojson'
with open(geojson_path, 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': cell_polygons}, f)
print(f'Saved GeoJSON → {geojson_path}')

# ── DuckDB — boundary_cells table ─────────────────────────────────────────
# Use register()+SELECT to call ST_GeomFromText() on a column — avoids the
# "ST_GeomFromText requires a string argument" error with executemany(?).
con = duckdb.connect(str(DB_PATH))
con.execute('LOAD spatial')

con.execute("""
    CREATE OR REPLACE TABLE boundary_cells (
        h3_id      VARCHAR PRIMARY KEY,
        resolution INTEGER,
        geometry   GEOMETRY
    )
""")

cells_df = pd.DataFrame([
    {
        'h3_id':        cell,
        'resolution':   RESOLUTION,
        'geometry_wkt': 'POLYGON ((' +
                         ', '.join(f'{lon} {lat}'
                                   for lon, lat in feat['geometry']['coordinates'][0][:-1]) +
                         ', ' + '{} {}'.format(*feat['geometry']['coordinates'][0][0]) + '))',
    }
    for cell, feat in zip(cells, cell_polygons)
])

con.register('_cells_temp', cells_df)
con.execute("""
    INSERT OR REPLACE INTO boundary_cells
    SELECT h3_id, resolution, ST_GeomFromText(geometry_wkt)
    FROM _cells_temp
""")
con.unregister('_cells_temp')

count = con.execute("SELECT count(*) FROM boundary_cells").fetchone()[0]
con.close()

print(f'Saved DuckDB  → boundary_cells  ({count} cells, resolution {RESOLUTION})  [{DB_PATH.name}]')
print()
print(pd.DataFrame({'h3_id': cells}).head(5).to_string(index=False))